# Deep Researcher Agent – Customization

This notebook walks through **customizing** the NVIDIA AI-Q Blueprint deep researcher: adding tools (paper search), assigning different models to the orchestrator vs planner/researcher, locating and editing prompts, and adding the knowledge layer. 

You will:

1. **Add a tool** – Install and configure the paper search tool (Google Scholar via Serper) and attach it to the deep research agent.
2. **Customize models** – Use a different LLM for the orchestrator (for example a frontier model) and Nemotron for the planner and researcher.
3. **Locate and customize prompts** – Find where orchestrator, planner, and researcher prompts live and what variables they use.
4. **Add the knowledge layer** – Enable document retrieval so the agent can search over your ingested documents alongside web and paper search.

# Table of Contents

>[Prerequisites](#prereqs)  
>[1. Adding a tool: Paper search](#add-tool)  
>[2. Customizing models (orchestrator vs planner/researcher)](#customize-models)  
>[3. Prompts: Where they are and how to customize](#prompts)  
>[4. Adding the knowledge layer](#knowledge-layer)    
>[Putting it together](#put-together)  
>[Next steps](#next)  

<a id="prereqs"></a>
## Prerequisites

This notebook assumes you have completed the environment setup from the [0_Getting_Started_with_AIQ](0_Getting_Started_with_AIQ.ipynb) and run through [1_Deep_Researcher_Web_Search](1_Deep_Researcher_Web_Search.ipynb).

**Ensure you have:**
- Cloned the repository and run `./scripts/setup.sh`
- Activated the virtual environment: `source .venv/bin/activate`
- API keys (you will be prompted in the next cells): **NVIDIA_API_KEY**, **TAVILY_API_KEY**; for paper search, **SERPER_API_KEY** from [serper.dev](https://serper.dev/)

In [ ]:
import os
from pathlib import Path

if os.environ.get("REPO_ROOT"):
    os.chdir(os.environ["REPO_ROOT"])
else:
    cwd = Path.cwd()
    if not (cwd / "configs").exists() and not (cwd / "src").exists():
        root = cwd.parent.parent
        if (root / "configs").is_dir() or (root / "src").is_dir():
            os.chdir(root)
print("Working directory:", os.getcwd())

In [ ]:
import getpass

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ:
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

<a id="add-tool"></a>
## 1. Adding a tool: Paper search

The deep researcher can use multiple search tools. The **paper search** tool queries academic papers through Google Scholar (Serper API). Adding it is a two-step process: set up a Serper API key, install the plugin, then add the function and tool list in your config.

**Reference:** [sources/google_scholar_paper_search/README.md](../../sources/google_scholar_paper_search/README.md), [Tools and Sources](../source/customization/tools-and-sources.md) (enable/disable by editing `tools` lists).

### 1.0 Set up Serper API key (required for paper search)

The paper search tool uses the [Serper](https://serper.dev/) API to query Google Scholar. You must have a Serper API key before installing or using the tool.

**Steps:**
1. Go to [serper.dev](https://serper.dev/) and create an account.
2. From your dashboard, generate an API key.
3. Add the key to your environment.

In [ ]:
import getpass

if "SERPER_API_KEY" not in os.environ:
    serper_api_key = getpass.getpass("Enter your Serper API key (or press Enter to skip): ")
    if serper_api_key:
        os.environ["SERPER_API_KEY"] = serper_api_key
        print("SERPER_API_KEY set for this session.")
    else:
        print("Skipped. Set SERPER_API_KEY in deploy/.env or run this cell again to use paper search.")

### 1.1 Install the paper search plugin

From the repo root, install the plugin so it registers the `paper_search` function type with the NeMo Agent toolkit:

In [ ]:
!uv pip install -e sources/google_scholar_paper_search

In [ ]:
!nat info components -t function 2>/dev/null | grep -E 'paper_search|paper_search_tool' || echo 'Run: nat info components -t function'

### 1.2 Add to your configuration

In your workflow YAML you need to:

1. **Define the function** under `functions:` with `_type: paper_search` and options such as `max_results`, `serper_api_key` (or rely on `SERPER_API_KEY` in the environment).
2. **Add the tool to the deep research agent** by including its name (for example `paper_search_tool`) in `deep_research_agent.tools`.

Example snippet:

```yaml
functions:
  paper_search_tool:
    _type: paper_search
    max_results: 5
    serper_api_key: ${SERPER_API_KEY}

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - paper_search_tool
      - advanced_web_search_tool
```

The agent’s prompts (orchestrator, planner, researcher) already mention `paper_search_tool` when that tool is present; they use the tool name to show the right instructions. No code changes are required beyond config.

In [ ]:
%%writefile docs/notebooks/config_deep_researcher_with_paper_search.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_deep:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    top_p: 1.0 
    max_tokens: 128000
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

functions:
  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 5
    advanced_search: true

  paper_search_tool:
    _type: paper_search
    max_results: 5
    serper_api_key: ${SERPER_API_KEY}

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - paper_search_tool
      - advanced_web_search_tool

workflow:
  _type: deep_research_workflow

### 1.3 Run with paper search (optional)

If you have `SERPER_API_KEY` set in `deploy/.env`, you can run the deep research workflow with paper search using the default CLI config (which already includes `paper_search_tool`). This may take several minutes.

In [ ]:
!nat run --config_file docs/notebooks/config_deep_researcher_with_paper_search.yml --input "Recent advances in transformer architecture for LLMs"

### 1.4 What to expect

When you run the workflow with paper search enabled, check the logs for tool invocations. You should see **`paper_search_tool`** (or `paper_search`) listed in tool-call messages as the planner and researcher run. For example: `[Tool Start] paper_search_tool` or `→ paper_search_tool` in the trace output. The agent may call it alongside `advanced_web_search_tool` for academic-style queries. If paper search is never called, the query may not have triggered academic search in the plan; try a more explicitly scholarly prompt (for example, "recent peer-reviewed advances in …").

<a id="customize-models"></a>
## 2. Customizing models (orchestrator vs planner/researcher)

The deep research agent has three LLM roles:

| Role | Config key | Purpose |
|------|------------|--------|
| **Orchestrator** | `orchestrator_llm` | Coordinates the workflow and writes the final report (required). |
| **Planner** | `planner_llm` | Builds the research plan and queries (optional; defaults to orchestrator). |
| **Researcher** | `researcher_llm` | Runs search and synthesizes content (optional; defaults to orchestrator). |

You can assign **different models** per role. For example: use a **frontier model** for the orchestrator (better report quality) and **Nemotron** for the planner and researcher (faster, lower cost).

**Reference:** [Configuration Reference](../source/customization/configuration-reference.md)

### 2.1 Example: Frontier orchestrator, Nemotron for planner and researcher

Define two (or three) LLMs in `llms:` and point the deep research agent at them:

```yaml
llms:
  nemotron_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    max_tokens: 128000

  frontier_llm:
    _type: openai
    model_name: "gpt-5.2"
    api_key: ${OPENAI_API_KEY}
    temperature: 1.0
    max_tokens: 128000

functions:
  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: frontier_llm 
    planner_llm: nemotron_llm
    researcher_llm: nemotron_llm
    max_loops: 2
    tools:
      - advanced_web_search_tool
```

If you omit `planner_llm` or `researcher_llm`, the agent falls back to `orchestrator_llm` for that role. The registration and wiring are in `src/aiq_agent/agents/deep_researcher/register.py` (it resolves each LLM ref and configures the shared `LLMProvider`).

<a id="prompts"></a>
## 3. Prompts: Where they are and how to customize

The deep researcher uses **Jinja2** templates for the orchestrator, planner, and researcher. Editing these files is the main way to change behavior (tone, structure, tool prioritization, or workflow steps).

**Reference:** [Deep Researcher README](../../src/aiq_agent/agents/deep_researcher/README.md) (Prompts section), `src/aiq_agent/agents/deep_researcher/agent.py` (`_load_prompts`, `render_prompt_template`).

### 3.1 Where the prompts live

All three prompts are under `src/aiq_agent/agents/deep_researcher/prompts/`:

| File | Role | Purpose |
|------|------|--------|
| `orchestrator.j2` | Orchestrator | System instructions: workflow steps, progress tracking, when to call planner/researcher, report format. |
| `planner.j2` | Planner subagent | Task analysis, TOC, constraints, query generation; tool prioritization (knowledge, paper, web). |
| `researcher.j2` | Researcher subagent | How to gather and synthesize information; tool use order; depth and citation requirements. |

At runtime the agent loads them with `load_prompt(AGENT_DIR / "prompts", name)` (see `agent.py`). No config switch is needed to “enable” a prompt; changing the file and restarting (or re-running) is enough.

In [ ]:
from pathlib import Path

prompts_dir = Path("src/aiq_agent/agents/deep_researcher/prompts")
for f in sorted(prompts_dir.glob("*.j2")):
    print(f.name)

### 3.2 Template variables

The templates are rendered with variables so the same prompt adapts to tools and context:

| Variable | Used in | Description |
|----------|--------|-------------|
| `tools` | All three | List of tool info (name, etc.); prompts use it to show “you have paper_search_tool, advanced_web_search_tool, …” and prioritization. |
| `current_datetime` | Orchestrator, researcher | Injected at render time for date-aware answers. |
| `clarifier_result` | Orchestrator | User clarifications from the clarifier agent, if any. |
| `available_documents` | Orchestrator, planner, researcher | User-uploaded docs (e.g. from knowledge layer) with summaries; prompts tell the agent to prioritize knowledge_search for these. |

The logic that builds these and calls `render_prompt_template` is in `agent.py` (`_build_orchestrator_agent`, `_get_subagents`). To add a new variable, add it to the `render_prompt_template(...)` call and use it in the corresponding `.j2` file.

### 3.3 How to customize

1. **Edit the `.j2` file** for the role you care about (orchestrator, planner, or researcher).
2. **Keep existing variables** – The templates use `{{ current_datetime }}`, `{% for tool in tools %}`, `available_documents`, etc. Removing or renaming these can break rendering.
3. **Adjust wording, steps, or priorities** – For example, in `researcher.j2` you can strengthen “cite sources inline” or add domain-specific guidance; in `orchestrator.j2` you can add or reorder workflow steps (as long as the tools and subagent names stay consistent).
4. **Restart or re-run** – The agent loads prompts at initialization; after saving changes, restart the process or re-run the notebook so the new content is used.

<a id="knowledge-layer"></a>
## 4. Adding the knowledge layer

The **knowledge layer** lets the agent search over **your ingested documents** (PDFs, docs, etc.) in addition to web and paper search. It is implemented as a NAT function type `knowledge_retrieval` with pluggable backends (LlamaIndex + ChromaDB for local dev, or Foundational RAG for a hosted server).

**Reference:** [Knowledge Layer](../source/customization/knowledge-layer.md), `sources/knowledge_layer/KNOWLEDGE-LAYER-SETUP.md` (if present), `configs/config_web_default_llamaindex.yml`.

### 4.1 Install a backend

Choose one backend and install it. For local development without extra services, use the LlamaIndex backend:

In [ ]:
# LlamaIndex backend (local ChromaDB; no external server)
!uv pip install -e "sources/knowledge_layer[llamaindex]"

In [ ]:
from aiq_agent.knowledge import get_retriever

print("Knowledge layer OK:", get_retriever is not None)

### 4.2 Add to configuration

In your YAML:

1. **Define the function** under `functions:` with `_type: knowledge_retrieval`, `backend: llamaindex` (or `foundational_rag`), plus backend-specific options (`collection_name`, `top_k`, `chroma_dir` for LlamaIndex; `rag_url` / `ingest_url` for Foundational RAG).
2. **Add the tool to the deep research agent** by including the function name (for example `knowledge_search`) in `deep_research_agent.tools`.

Example for LlamaIndex (see `configs/config_web_default_llamaindex.yml`):

```yaml
functions:
  knowledge_search:
    _type: knowledge_retrieval
    backend: llamaindex
    collection_name: ${COLLECTION_NAME:-test_collection}
    top_k: 5
    chroma_dir: ${AIQ_CHROMA_DIR:-/tmp/chroma_data}

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - advanced_web_search_tool
      - knowledge_search
```

The deep researcher prompts already include branches for `knowledge_search` (and `available_documents`); once the tool is in the list, the orchestrator and subagents will be instructed to use it for user documents and internal context.

In [ ]:
%%writefile docs/notebooks/config_deep_researcher_with_knowledge_layer.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_deep:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    top_p: 1.0
    max_tokens: 128000
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

functions:
  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 5
    advanced_search: true

  knowledge_search:
    _type: knowledge_retrieval
    backend: llamaindex
    collection_name: ${COLLECTION_NAME:-test_collection}
    top_k: 5
    chroma_dir: ${AIQ_CHROMA_DIR:-/tmp/chroma_data}

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - advanced_web_search_tool
      - knowledge_search

workflow:
  _type: deep_research_workflow

<a id="put-together"></a>
## Putting it together

A single config can combine all of the above:

- **Tools:** `paper_search_tool`, `advanced_web_search_tool`, `knowledge_search` in `deep_research_agent.tools`.
- **Models:** `orchestrator_llm: frontier_llm`, `planner_llm: nemotron_llm`, `researcher_llm: nemotron_llm` (with those LLMs defined in `llms:`).
- **Prompts:** No config change; edit `src/aiq_agent/agents/deep_researcher/prompts/*.j2` as needed.
- **Knowledge layer:** Install a backend, define `knowledge_search` in `functions`, and add it to `deep_research_agent.tools`.

See `configs/config_web_default_llamaindex.yml` and `configs/config_web_frag.yml` for full examples that mix web search, paper search, knowledge layer, and different LLMs.

<a id="next"></a>
## Next steps

- **Guide to Extending** (see [Extending Documentation](../source/extending/index.md))